In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
from torch import nn
import torch

In [9]:
class Net(nn.Module):
  def __init__(self, *args):
    self.r = nn.Parameter(data=torch.tensor([0.]))

def physics_loss_discovery(model: torch.nn.Module):
    ts = torch.linspace(0, 1000, steps=1000,).view(-1,1).requires_grad_(True).to(DEVICE)
    temps = model(ts)
    dT = grad(temps, ts)[0]
    # use the differentiable parameter instead
    pde = model.r * (Tenv - temps) - dT
    
    return torch.mean(pde**2)

In [10]:
def grad(outputs, inputs):
    """Computes the partial derivative of 
    an output with respect to an input."""
    return torch.autograd.grad(
        outputs, 
        inputs, 
        grad_outputs=torch.ones_like(outputs), 
        create_graph=True
    )

def physics_loss(model: torch.nn.Module):
    """The physics loss of the model"""
    # make collocation points
    ts = torch.linspace(0, 1000, steps=1000,).view(-1,1).requires_grad_(True)
    # run the collocation points through the network
    temps = model(ts)
    # get the gradient
    dT = grad(temps, ts)[0]
    # compute the ODE
    ode = dT - R*(Tenv - temps)
    # MSE of ODE
    return torch.mean(ode**2)

In [11]:
def plot_losses(train_losses, train_metrics, val_losses, val_metrics):
    """
    Plot losses and metrics while training
      - train_losses: sequence of train losses
      - train_metrics: sequence of train MSE values
      - val_losses: sequence of validation losses
      - val_metrics: sequence of validation MSE values
    """
    clear_output()
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    axs[0].plot(range(1, len(train_losses) + 1), train_losses, label="train")
    axs[0].plot(range(1, len(val_losses) + 1), val_losses, label="val")
    axs[1].plot(range(1, len(train_metrics) + 1), train_metrics, label="train")
    axs[1].plot(range(1, len(val_metrics) + 1), val_metrics, label="val")

    if max(train_losses) / min(train_losses) > 10:
        axs[0].set_yscale("log")

    if max(train_metrics) / min(train_metrics) > 10:
        axs[0].set_yscale("log")

    for ax in axs:
        ax.set_xlabel("epoch")
        ax.legend()

    axs[0].set_ylabel("loss")
    axs[1].set_ylabel("MSE")
    plt.show()


def train_and_validate(
    model,
    optimizer,
    criterion,
    metric,
    train_loader,
    val_loader,
    num_epochs,
    verbose=True,
):
    """
    Train and validate neural network
      - model: neural network to train
      - optimizer: optimizer chained to a model
      - criterion: loss function class
      - metric: function to measure MSE taking neural networks predictions
                 and ground truth labels
      - train_loader: DataLoader with train set
      - val_loader: DataLoader with validation set
      - num_epochs: number of epochs to train
      - verbose: whether to plot metrics during training
    Returns:
      - train_mse: training MSE over the last epoch
      - val_mse: validation MSE after the last epoch
    """
    train_losses, val_losses = [], []
    train_metrics, val_metrics = [], []

    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss, running_metric = 0, 0
        pbar = (
            tqdm(train_loader, desc=f"Training {epoch}/{num_epochs}")
            if verbose
            else train_loader
        )

        for i, (X_batch, y_batch) in enumerate(pbar, 1):
            predictions = model.forward(X_batch)
            loss = criterion(predictions, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            with torch.no_grad():
                metric_value = metric(predictions, y_batch)
                if type(metric_value) == torch.Tensor:
                    metric_value = metric_value.item()
                running_loss += loss.item() * X_batch.shape[0]
                running_metric += metric_value * X_batch.shape[0]

            if verbose and i % 100 == 0:
                pbar.set_postfix({"loss": loss.item(), "MSE": metric_value})

        train_losses += [running_loss / len(train_loader.dataset)]
        train_metrics += [running_metric / len(train_loader.dataset)]

        model.eval()
        running_loss, running_metric = 0, 0
        pbar = (
            tqdm(val_loader, desc=f"Validating {epoch}/{num_epochs}")
            if verbose
            else val_loader
        )

        for i, (X_batch, y_batch) in enumerate(pbar, 1):
            with torch.no_grad():
                predictions = model.forward(X_batch)
                loss = criterion(predictions, y_batch)
                
                metric_value = metric(predictions, y_batch)
                if type(metric_value) == torch.Tensor:
                    metric_value = metric_value.item()
                running_loss += loss.item() * X_batch.shape[0]
                running_metric += metric_value * X_batch.shape[0]

            if verbose and i % 100 == 0:
                pbar.set_postfix({"loss": loss.item(), "MSE": metric_value})
        val_losses += [running_loss / len(val_loader.dataset)]
        val_metrics += [running_metric / len(val_loader.dataset)]

        if verbose:
            plot_losses(train_losses, train_metrics, val_losses, val_metrics)

    if verbose:
        print(f"Validation MSE: {val_metrics[-1]:.3f}")

    return train_metrics[-1], val_metrics[-1]

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
df = pd.read_csv("../Sem_3/ML/dl-course-hse-masters-mds2526/homeworks/HW1/YearPredictionMSD.txt.zip", header=None)
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

train_size = int(0.75 * X.shape[0])
X_train_raw = X[:train_size, :]
y_train_raw = y[:train_size]
X_test = X[train_size:, :]
y_test = y[train_size:]
X_train_raw.shape, X_test.shape

X_train, X_val, y_train, y_val = train_test_split(
    X_train_raw, y_train_raw, test_size=0.25, random_state=0xE2E4
)
X_train.shape, X_val.shape

In [ ]:
from sklearn.preprocessing import LabelEncoder
n_hidden = 128

le = LabelEncoder()
y_target_train = le.fit_transform(y_train)
y_target_val = le.transform(y_val)

n_in, n_out = 90, len(le.classes_)
model = nn.Sequential(nn.Linear(n_in, n_hidden), nn.ReLU(), 
                      nn.Linear(n_hidden, n_out)).to(torch.float32)

optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)

criterion = nn.CrossEntropyLoss()
metric = lambda x,y: nn.MSELoss()(torch.argmax(x, dim=1).float(),y.float())
num_epochs = 4
batch_size = 64

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.LongTensor(y_target_train))
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size)
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.LongTensor(y_target_val))
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size)

train_and_validate(model, optimizer, criterion, metric, train_loader, val_loader, num_epochs)